# BDA Practica 2 — Data Pipeline Validation

This notebook validates the end-to-end Data Engineering pipeline (Landing → Formatted → Trusted → Exploitation) for the three active sources: **Sulianova cardiovascular disease**, **CDC heart disease health indicators**, and **Cleveland heart disease**.

It complements the two analytical notebooks:

1. `01_data_pipeline_validation.ipynb` — *this notebook*: data quality and integration of the trusted and exploitation tables.
2. `02_knowledge_graph_exploration.ipynb` — explores the RDF/RDFS Knowledge Graph and runs the SPARQL analytical queries.
3. `03_model_comparison_and_explainability.ipynb` — compares the two integrated ML models with the KG-embedding ML model and explains them.

Run the full pipeline first (`python run_all_pipeline.py --skip-landing --strict`) and then open these notebooks.

In [1]:
from pathlib import Path
import duckdb
import pandas as pd

def locate_project_root(start: Path) -> Path:
    current = start.resolve()
    for _ in range(8):
        if (current / 'run_all_pipeline.py').exists():
            return current
        current = current.parent
    raise FileNotFoundError('Could not locate project root')

PROJECT_ROOT = locate_project_root(Path.cwd())
TRUSTED_DB = PROJECT_ROOT / 'Part3_Trusted_zone' / 'trusted_zone' / 'trusted.duckdb'
EXP_DB = PROJECT_ROOT / 'Part4_Exploitation_zone' / 'exploitation_zone' / 'exploitation.duckdb'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('TRUSTED_DB exists:', TRUSTED_DB.exists())
print('EXP_DB exists:', EXP_DB.exists())

PROJECT_ROOT: /home/stargix/Desktop/uni/BDA/bda-practica1
TRUSTED_DB exists: True
EXP_DB exists: True


In [2]:
con_t = duckdb.connect(str(TRUSTED_DB))
con_e = duckdb.connect(str(EXP_DB))

trusted_counts = con_t.execute('''
SELECT 'trusted.cardiovascular_disease' AS table_name, COUNT(*) AS rows FROM trusted.trusted.cardiovascular_disease
UNION ALL
SELECT 'trusted.heart_disease_health_indicators', COUNT(*) FROM trusted.trusted.heart_disease_health_indicators
UNION ALL
SELECT 'trusted.heart_disease_cleveland', COUNT(*) FROM trusted.trusted.heart_disease_cleveland
UNION ALL
SELECT 'metadata.quality_report', COUNT(*) FROM trusted.metadata.quality_report
UNION ALL
SELECT 'metadata.quarantine', COUNT(*) FROM trusted.metadata.quarantine
''').df()

exploitation_counts = con_e.execute('''
SELECT 'exploitation.risk_model_input' AS table_name, COUNT(*) AS rows FROM exploitation.exploitation.risk_model_input
UNION ALL
SELECT 'exploitation.dataset_profile', COUNT(*) FROM exploitation.exploitation.dataset_profile
UNION ALL
SELECT 'metadata.metadata_lineage', COUNT(*) FROM exploitation.metadata.metadata_lineage
''').df()

trusted_counts, exploitation_counts

(                                table_name    rows
 0           trusted.cardiovascular_disease   68610
 1  trusted.heart_disease_health_indicators  253680
 2          trusted.heart_disease_cleveland     297
 3                  metadata.quality_report      54
 4                      metadata.quarantine    1390,
                       table_name    rows
 0  exploitation.risk_model_input  322587
 1   exploitation.dataset_profile       3
 2      metadata.metadata_lineage       1)

In [3]:
quality_top = con_t.execute('''
SELECT dataset, rule_name, violations, violation_pct, action_applied
FROM trusted.metadata.quality_report
ORDER BY violations DESC, dataset
LIMIT 20
''').df()

quarantine_top = con_t.execute('''
SELECT dataset, quarantine_reason, COUNT(*) AS rows
FROM trusted.metadata.quarantine
GROUP BY 1,2
ORDER BY rows DESC, dataset
LIMIT 15
''').df()

quality_top, quarantine_top

(                    dataset                       rule_name  violations  \
 0    cardiovascular_disease          rows_quarantined_total        1390   
 1    cardiovascular_disease    systolic_bp_gte_diastolic_bp        1234   
 2    cardiovascular_disease     diastolic_bp_between_40_150        1034   
 3    cardiovascular_disease      systolic_bp_between_70_250         229   
 4    cardiovascular_disease       height_cm_between_120_250          52   
 5    cardiovascular_disease               bmi_between_10_80          39   
 6    cardiovascular_disease        weight_kg_between_30_300           7   
 7    cardiovascular_disease              patient_id_present           0   
 8    cardiovascular_disease        age_years_between_18_100           0   
 9    cardiovascular_disease     age_group_code_between_1_13           0   
 10   cardiovascular_disease       gender_domain_male_female           0   
 11   cardiovascular_disease  cholesterol_level_domain_1_2_3           0   
 12   cardio

In [4]:
profile = con_e.execute('''
SELECT source_dataset, rows_total, target_positive_rate, age_proxy_avg, bmi_avg,
       high_bp_rate, high_chol_rate, bmi_missing_pct, smoking_missing_pct, activity_missing_pct
FROM exploitation.exploitation.dataset_profile
ORDER BY source_dataset
''').df()

profile

,source_dataset,rows_total,target_positive_rate,age_proxy_avg,bmi_avg,high_bp_rate,high_chol_rate,bmi_missing_pct,smoking_missing_pct,activity_missing_pct
0,cardiovascular_disease,68610,0.4947,52.792,27.454,0.8156,0.2501,0.0,0.0,0.0
1,heart_disease_cleveland,297,0.4613,54.542,NaN,0.5556,0.8384,100.0,100.0,100.0
2,heart_disease_health_indicators,253680,0.0942,57.138,28.382,0.4290,0.4241,0.0,0.0,0.0


In [ ]:
import matplotlib.pyplot as plt

quality_summary = con_t.execute('''
SELECT dataset,
       SUM(violations) FILTER (WHERE rule_name <> 'rows_quarantined_total') AS rule_violations,
       MAX(CASE WHEN rule_name = 'rows_quarantined_total' THEN violations END) AS quarantined_rows,
       MAX(total_rows) AS source_rows
FROM trusted.metadata.quality_report
GROUP BY dataset
ORDER BY dataset
''').df()
quality_summary["quarantine_pct"] = (
    quality_summary["quarantined_rows"].fillna(0) / quality_summary["source_rows"].clip(lower=1) * 100
)

fig, ax = plt.subplots(figsize=(8, 3.2), constrained_layout=True)
bars = ax.barh(quality_summary["dataset"], quality_summary["quarantine_pct"], color="#4F8DCB")
ax.set_xlabel("% of rows quarantined")
ax.set_title("Trusted Zone quarantine rate per dataset")
for bar, pct in zip(bars, quality_summary["quarantine_pct"]):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height() / 2,
            f"{pct:.2f}%", va="center")
plt.show()
quality_summary

In [ ]:
coverage = con_e.execute('''
SELECT source_dataset,
       100.0 * AVG(CASE WHEN bmi IS NULL THEN 0 ELSE 1 END) AS bmi_coverage,
       100.0 * AVG(CASE WHEN systolic_bp IS NULL THEN 0 ELSE 1 END) AS systolic_bp_coverage,
       100.0 * AVG(CASE WHEN smoking_flag IS NULL THEN 0 ELSE 1 END) AS smoking_coverage,
       100.0 * AVG(CASE WHEN high_blood_pressure_flag IS NULL THEN 0 ELSE 1 END) AS high_bp_coverage,
       100.0 * AVG(CASE WHEN high_cholesterol_flag IS NULL THEN 0 ELSE 1 END) AS high_chol_coverage,
       100.0 * AVG(CASE WHEN max_heart_rate IS NULL THEN 0 ELSE 1 END) AS max_hr_coverage,
       100.0 * AVG(CASE WHEN general_health_score IS NULL THEN 0 ELSE 1 END) AS general_health_coverage
FROM exploitation.exploitation.risk_model_input
GROUP BY source_dataset
ORDER BY source_dataset
''').df()
coverage

## Interpretation notes

- Trusted now stores three active datasets: Sulianova, CDC, and Cleveland.
- Quality metrics should be read per source: the three datasets have very different attribute spaces, so the violation rules and missingness profiles are not directly comparable.
- The exploitation `dataset_profile` shows the reconciled feature coverage and missingness for the integrated cross-source schema. Columns that are missing in a source appear as 100% `*_missing_pct` for that source, which is the expected behaviour of the integration: no false person-level merging is performed.
- The Cleveland source is intentionally small (~297 rows) but provides clinical features (max heart rate, exercise-induced angina) that are absent in the other two larger sources.
- The integration uses **shared semantic concepts** (age band, gender, indicators, outcomes) rather than a flat foreign-key merge — this is what enables the Knowledge Graph in Part4 and the SPARQL/graph-embedding analysis in Part5.